In [1]:
!pip install transformers torch sentencepiece --quiet


In [3]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

print("Loading FLAN-T5-large...")
model_name = "google/flan-t5-large"
tokenizer  = T5Tokenizer.from_pretrained(model_name)
model_t5   = T5ForConditionalGeneration.from_pretrained(model_name)
model_t5   = model_t5.to(device)

def llm(prompt, max_new_tokens=100):
    inputs  = tokenizer(prompt, return_tensors="pt",
                        truncation=True, max_length=512).to(device)
    outputs = model_t5.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,           # beam search = better quality
        early_stopping=True,
        no_repeat_ngram_size=3 # prevents repetition
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return [{"generated_text": text}]

print(" FLAN-T5-large loaded!")

# Test with a proper summarisation prompt
# Test with more explicit instruction format


   Device: cuda
Loading FLAN-T5-large...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


 FLAN-T5-large loaded!
   Test: The package arrived 2 weeks late | Delivery took FOREVER


In [4]:
test = llm("""Instruction: Write a one sentence summary of the delivery problems mentioned.
Input: The package arrived 2 weeks late. Delivery took forever. Item came damaged due to long transit. Very disappointed with shipping time.
Output:""")
print(f"Test: {test[0]['generated_text']}")

Test: The package arrived 2 weeks late. Delivery took FOREVER.


In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd

# Reload Gold layer
df_gold = pd.read_csv("/content/drive/MyDrive/olist_project/gold/predictions.csv")
print(f" Gold loaded — {len(df_gold):,} orders")

# Test BART with real reviews
real_reviews = df_gold[
    df_gold["review_comment_message"].notna()
]["review_comment_message"].head(20).tolist()

combined = " ".join([str(r) for r in real_reviews if len(str(r)) > 20])[:1000]
print(f"\n📝 Input text ({len(combined)} chars):")
print(combined[:300] + "...")

print("\n Summarising...")
test = llm(combined)
print(f"\nSummary: {test[0]['generated_text']}")


Mounted at /content/drive
 Gold loaded — 96,470 orders

📝 Input text (1000 chars):
Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente. O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista. Aguardando retorno da l...

 Summarising...

Summary: O produço incrivel, muiro macios e super em conta


In [14]:
def extractive_summary(reviews: list, n: int = 2) -> str:
    """Pick longest, most informative reviews as summary."""
    clean = [r.strip() for r in reviews
             if isinstance(r, str) and len(r.strip()) > 30][:50]
    if not clean:
        return "No reviews available."

    # Sort by length — longer reviews = more informative
    scored = sorted(clean, key=len, reverse=True)

    # Return top 2 longest reviews trimmed to 150 chars each
    top = scored[:n]
    return " | ".join([r[:150] for r in top])

print("Generating category summaries...\n")
category_summaries = {}
top_cats = df_gold["product_category_name_english"].value_counts().head(10).index.tolist()

for cat in top_cats:
    reviews = df_gold[
        df_gold["product_category_name_english"] == cat
    ]["review_comment_message"].dropna().tolist()

    summary = extractive_summary(reviews)
    category_summaries[cat] = summary
    print(f" {cat}")
    print(f"   → {summary[:200]}\n")

print(f"Done — {len(category_summaries)} categories summarised!")

Generating category summaries...

 bed_bath_table
   → Acho um absurdo comprar um produto que diz estar disponível e não recebê-lo !!! Verifiquem por favor, pois não é a primeira vez que acontece. Exijo um | Gostei bastante, chegou antes do previsto, suge

 health_beauty
   → Como o produto foi violado, já entrei em contato com o baratheon que segundo a atendente será reenviado outro quite.Quero salientar ainda que o único  | gostei muito de compra com voces, so gostaria d

 sports_leisure
   → Recebi apenas 1 unidade solicitada, deveriam ser 2, inclusive foi cobrado e consta na nota fiscal. Estou tentando contato com a loja a 4 dias. Informa | Não houve competência na entrega ou seja, simpl

 computers_accessories
   → Preciso alizar a troca do produto que está com defeito e não consigo registrar a solicitação no SITE. No item "TROCAS E DEVOLUÇÕES" sempre informa que | O pedido, não chegou com a quantidade correta. 

 furniture_decor
   → O site do stark consta meu pedido como totalme

In [17]:
# Reload Silver to get feature columns back
df_silver = pd.read_csv("/content/drive/MyDrive/olist_project/silver/features.csv")

# Merge features back into Gold
df_gold_full = df_gold.merge(
    df_silver[["order_id", "seller_late_rate", "promised_days", "cross_state",
               "carrier_days", "heavy_item", "is_holiday_season",
               "payment_installments", "is_weekend_order"]],
    on="order_id", how="left"
)

print(f"✅ Merged! Columns now: {len(df_gold_full.columns)}")
print(f"   Sample: {list(df_gold_full.columns)}")

# Verify features are there
sample = df_gold_full[df_gold_full["risk_tier"] == "Critical"].iloc[0]
print(f"\nSample critical row:")
print(sample[["late_probability","seller_late_rate","promised_days",
              "cross_state","carrier_days"]])

✅ Merged! Columns now: 16
   Sample: ['order_id', 'target_late', 'review_comment_message', 'product_category_name_english', 'late_probability', 'predicted_late', 'risk_tier', 'revenue_at_risk', 'seller_late_rate', 'promised_days', 'cross_state', 'carrier_days', 'heavy_item', 'is_holiday_season', 'payment_installments', 'is_weekend_order']

Sample critical row:
late_probability      0.9854
seller_late_rate    0.666667
promised_days              9
cross_state                0
carrier_days            17.0
Name: 19, dtype: object


In [19]:
# Debug — print exact values for first critical order
row = high_risk.iloc[0]
print("=== DEBUG: First Critical Order ===")
print(f"seller_late_rate:    {row.get('seller_late_rate')} (need > 0.3)")
print(f"promised_days:       {row.get('promised_days')} (need < 7)")
print(f"cross_state:         {row.get('cross_state')} (need == 1)")
print(f"carrier_days:        {row.get('carrier_days')} (need > 5)")
print(f"heavy_item:          {row.get('heavy_item')} (need == 1)")
print(f"is_holiday_season:   {row.get('is_holiday_season')} (need == 1)")
print(f"payment_installments:{row.get('payment_installments')} (need > 6)")
print(f"is_weekend_order:    {row.get('is_weekend_order')} (need == 1)")
print(f"\nlate_probability:    {row.get('late_probability')}")
print(f"risk_tier:           {row.get('risk_tier')}")

=== DEBUG: First Critical Order ===
seller_late_rate:    None (need > 0.3)
promised_days:       None (need < 7)
cross_state:         None (need == 1)
carrier_days:        None (need > 5)
heavy_item:          None (need == 1)
is_holiday_season:   None (need == 1)
payment_installments:None (need > 6)
is_weekend_order:    None (need == 1)

late_probability:    0.9854
risk_tier:           Critical


In [20]:
# Check order_id format in both dataframes
print("Gold order_id sample:")
print(df_gold_full["order_id"].head(3).tolist())
print(f"dtype: {df_gold_full['order_id'].dtype}")

print("\nSilver order_id sample:")
print(df_silver["order_id"].head(3).tolist())
print(f"dtype: {df_silver['order_id'].dtype}")

# Check if any match at all
matches = df_gold_full["order_id"].isin(df_silver["order_id"]).sum()
print(f"\nMatching order_ids: {matches} / {len(df_gold_full)}")

Gold order_id sample:
['e481f51cbdc54678b7cc49136f2d6af7', '53cdb2fc8bc7dce0b6741e2150273451', '47770eb9100c2d0c44946d9cf07ec65d']
dtype: object

Silver order_id sample:
['e481f51cbdc54678b7cc49136f2d6af7', '53cdb2fc8bc7dce0b6741e2150273451', '47770eb9100c2d0c44946d9cf07ec65d']
dtype: object

Matching order_ids: 96470 / 96470


In [21]:
# Recreate high_risk from the MERGED dataframe
high_risk = df_gold_full[
    df_gold_full["risk_tier"].isin(["Critical", "High"])
].copy().reset_index(drop=True)

print(f"High risk orders: {len(high_risk)}")

# Debug first row again
row = high_risk.iloc[0]
print(f"\nseller_late_rate:    {row['seller_late_rate']}")
print(f"promised_days:       {row['promised_days']}")
print(f"carrier_days:        {row['carrier_days']}")
print(f"cross_state:         {row['cross_state']}")

# Generate explanations
high_risk["llm_explanation"] = high_risk.apply(explain_prediction, axis=1)

print("\n Sample explanations:\n")
for _, row in high_risk.head(5).iterrows():
    print(f"Order: {row['order_id'][:12]}...")
    print(f"→ {row['llm_explanation']}")
    print()

High risk orders: 4022

seller_late_rate:    0.6666666666666666
promised_days:       9
carrier_days:        17.0
cross_state:         0

 Sample explanations:

Order: 203096f03d82...
→ [CRITICAL RISK — 99% late probability] Flagged because: seller has 67% historical late rate; carrier took 17 days to dispatch. Recommended action: Contact customer proactively and upgrade shipping.

Order: fbf9ac61453a...
→ [HIGH RISK — 72% late probability] Flagged because: cross-state shipment adds transit risk. Recommended action: Contact customer proactively and upgrade shipping.

Order: 1790eea0b567...
→ [HIGH RISK — 69% late probability] Flagged because: cross-state shipment adds transit risk. Recommended action: Contact customer proactively and upgrade shipping.

Order: 6ea2f835b455...
→ [CRITICAL RISK — 97% late probability] Flagged because: cross-state shipment adds transit risk; carrier took 18 days to dispatch; heavy item increases logistics complexity; holiday season causes carrier overload; 

In [23]:
# Add category summaries
df_gold_full["category_review_summary"] = df_gold_full[
    "product_category_name_english"
].map(category_summaries).fillna("No summary available")

# Add explanations for all orders
df_gold_full["llm_explanation"] = df_gold_full.apply(
    explain_prediction, axis=1
)

# Final summary
print(" FINAL GOLD LAYER SUMMARY")
print(f"   Total orders:          {len(df_gold_full):,}")
print(f"   Predicted late:        {df_gold_full['predicted_late'].sum():,}")
print(f"   Critical risk:         {(df_gold_full['risk_tier']=='Critical').sum():,}")
print(f"   High risk:             {(df_gold_full['risk_tier']=='High').sum():,}")
print(f"   Total revenue at risk: ${df_gold_full['revenue_at_risk'].sum():,.0f}")
print(f"   Orders with reasons:   {(df_gold_full['llm_explanation'].str.contains('Flagged because: m')==False).sum():,}")

# Save final Gold
df_gold_full.to_csv(
    "/content/drive/MyDrive/olist_project/gold/intelligence_final.csv",
    index=False
)
print("\nFinal Gold intelligence layer saved!")
print("   File: olist_project/gold/intelligence_final.csv")

 FINAL GOLD LAYER SUMMARY
   Total orders:          96,470
   Predicted late:        5,293
   Critical risk:         1,745
   High risk:             2,277
   Total revenue at risk: $240,761
   Orders with reasons:   80,115

Final Gold intelligence layer saved!
   File: olist_project/gold/intelligence_final.csv
